# 02. Embeddings and Similarity

This notebook shows how short archive-like passages become vectors and why embedding quality matters for retrieval.

We will:
- encode a small corpus
- compare similarity scores for multiple queries
- inspect nearest neighbors
- visualize rough embedding neighborhoods

The main lesson is that semantic similarity can help retrieval, but it can also fail when the model does not understand the domain, the community terminology, or mixed-language inputs.


The workshop treats the system as retrieval-first:
- tokenization defines what the model sees
- retrieval quality often controls answer quality
- generators cannot reliably compensate for weak retrieval
- evaluation should include groundedness, citation, abstention, bilingual behavior, and community-review placeholders

Current notebook: `02_embeddings_and_similarity.ipynb`

This notebook turns archive-like passages into embeddings so participants can inspect nearest neighbors, semantic drift, and bilingual retrieval behavior.

This is a core workshop notebook.

Workshop sequence:
1. `01_tokenization_playground.ipynb`
2. `02_embeddings_and_similarity.ipynb`
3. `03_retriever_benchmarking_for_rag.ipynb`
4. `04_llm_comparison_in_same_rag_pipeline.ipynb`
5. `05_evaluating_rag_systems.ipynb`
6. `06_optional_lora_or_domain_adaptation.ipynb` (optional)


## Quick concept refresher

- Tokenization turns text into units that models can process.
- Retrieval chooses which passages are available as evidence.
- The retriever selects context; the generator turns context into an answer.
- Evaluation in archive assistants is multi-layered because the system must be useful, grounded, reviewable, and appropriate.


In [ ]:
# Self-contained runtime setup for this notebook.
# Each notebook includes its own seed and device checks so it can run independently in Colab.

import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")
print(f"Detected device: {DEVICE}")
print(f"Seed set to: {SEED}")

NOTEBOOK_CONTEXT = {
    "seed": SEED,
    "device": DEVICE,
    "notebook": __name__ if "__name__" in globals() else "notebook",
    "framing": "retrieval-first archive assistant workshop",
}

NOTEBOOK_CONTEXT


In [ ]:
# In Colab, uncomment the line below if sentence-transformers is missing.

# !pip -q install sentence-transformers scikit-learn pandas matplotlib seaborn

print("Embedding notebook dependencies are ready once the required packages are installed.")


In [ ]:
# Imports.

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

sns.set_theme(style="whitegrid")


In [ ]:
# A small archive-like corpus.
# The passages are deliberately short so the similarity behavior is easy to inspect.

corpus = [
    {
        "doc_id": "P01",
        "language": "en",
        "title": "Public metadata policy",
        "text": "Public descriptions may be shared openly, but restricted ceremonial material requires a community access review before release.",
    },
    {
        "doc_id": "P02",
        "language": "en",
        "title": "Place-name interview",
        "text": "The recording explains how elders prefer historical place-names to appear with their locally used spelling variants.",
    },
    {
        "doc_id": "P03",
        "language": "fr",
        "title": "Résumé bilingue",
        "text": "Le résumé bilingue décrit une entrevue sur les noms de lieux, les variantes orthographiques et les règles de citation.",
    },
    {
        "doc_id": "P04",
        "language": "en",
        "title": "Winter storytelling protocol",
        "text": "Several stories are marked for winter-only teaching sessions and should not be used outside the approved seasonal context.",
    },
    {
        "doc_id": "P05",
        "language": "en",
        "title": "Audio transcription note",
        "text": "This note compares manual transcription with OCR-derived transcript text and warns that speaker turns were merged incorrectly.",
    },
    {
        "doc_id": "P06",
        "language": "mixed",
        "title": "Catalog note with community term",
        "text": "The catalog entry links an English description with a community-language keyword placeholder for a kinship-based access practice.",
    },
]

corpus_df = pd.DataFrame(corpus)
corpus_df


In [ ]:
# Editable queries.
# Add your own queries to see how the embedding model behaves.

queries = [
    "Which material requires access review before public release?",
    "How should place-names and spelling variants be cited?",
    "Quels documents parlent des variantes orthographiques?",
    "Which records are limited to winter teaching sessions?",
    "Find mixed-language metadata about kinship-based access.",
]

queries


In [ ]:
# Load a compact multilingual embedding model.
# Multilingual models are useful for bilingual or mixed-language retrieval experiments.

embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(embedding_model_name)
print(f"Loaded embedding model: {embedding_model_name}")


In [ ]:
# Encode the corpus and the queries.
# normalize_embeddings=True is convenient because cosine similarity then becomes a dot product.

corpus_texts = corpus_df["text"].tolist()
corpus_embeddings = embedding_model.encode(
    corpus_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_embeddings = embedding_model.encode(
    queries,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Corpus embedding shape:", corpus_embeddings.shape)
print("Query embedding shape:", query_embeddings.shape)


In [ ]:
# Rank passages for each query by cosine similarity.
# This is the simplest dense retrieval baseline.

def rank_passages(query: str, query_embedding, corpus_df: pd.DataFrame, corpus_embeddings, top_k: int = 3):
    scores = cosine_similarity([query_embedding], corpus_embeddings)[0]
    ranked = corpus_df.copy()
    ranked["score"] = scores
    ranked = ranked.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)
    ranked.insert(0, "query", query)
    return ranked


ranked_results = []
for query, query_embedding in zip(queries, query_embeddings):
    ranked_results.append(rank_passages(query, query_embedding, corpus_df, corpus_embeddings, top_k=3))

ranked_df = pd.concat(ranked_results, ignore_index=True)
ranked_df


In [ ]:
# Inspect one query at a time in a compact display.
# This makes it easier to talk through good matches and surprising false positives.

focus_query = queries[0]
focus_embedding = query_embeddings[0]
focus_results = rank_passages(focus_query, focus_embedding, corpus_df, corpus_embeddings, top_k=5)

print("Focus query:", focus_query)
display(focus_results[["doc_id", "language", "title", "score", "text"]])


In [ ]:
# Visualize the embedding neighborhood with PCA.
# This is only a rough projection, but it helps participants see clustering behavior.

labels = [f"{row.doc_id}: {row.title}" for row in corpus_df.itertuples()]
projection = PCA(n_components=2, random_state=42).fit_transform(corpus_embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(projection[:, 0], projection[:, 1], s=100)

for (x, y), label in zip(projection, labels):
    plt.text(x + 0.01, y + 0.01, label, fontsize=9)

plt.title("Rough 2D view of archive-passage embedding neighborhoods")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.tight_layout()
plt.show()


In [ ]:
# Compare semantically similar versus lexically similar items.
# This exercise is useful when discussing why dense retrieval can help beyond keyword overlap.

comparison_queries = [
    "seasonal teaching restrictions",
    "winter-only story access",
    "orthographic variants in place names",
    "speaker turns merged in transcript",
]

comparison_embeddings = embedding_model.encode(
    comparison_queries,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

comparison_tables = []
for query, query_embedding in zip(comparison_queries, comparison_embeddings):
    table = rank_passages(query, query_embedding, corpus_df, corpus_embeddings, top_k=3)
    comparison_tables.append(table)

display(pd.concat(comparison_tables, ignore_index=True))


## Discussion prompts

- Which queries worked well even when lexical overlap was weak?
- Where did the model confuse topical similarity with the actual information need?
- Did the bilingual query behave as expected?
- How might a general embedding model fail on community-specific terminology or underrepresented orthographies?


In [ ]:
# Exercise cell: edit the corpus or queries and rerun the encoding section.
# This is intentionally lightweight so participants can experiment in live sessions.

custom_query = "Find records about citation rules for place-names."
custom_query_embedding = embedding_model.encode([custom_query], convert_to_numpy=True, normalize_embeddings=True)[0]
custom_results = rank_passages(custom_query, custom_query_embedding, corpus_df, corpus_embeddings, top_k=3)
display(custom_results)
